# 🛡️ Autonomous Network Intrusion Detection System (NIDS 2.0)
## Dual-Tier AI Architecture: Supervised LightGBM + Unsupervised Deep Autoencoder

**Authors:** Xebia Project Showcase  
**Schema Standard:** CICIDS2017 Benchmark Flow Schema (90 Features)  

---

### 🎯 What does this Notebook do?
In traditional cybersecurity, firewalls fail when a **Zero-Day (unseen attack)** arrives. This notebook trains an end-to-end **Dual-Tier AI defense**:
1. **Tier 1 (Supervised - LightGBM):** Classifies known attack signatures (`DoS`, `PortScan`, `BruteForce`) at microsecond latency.
2. **Tier 2 (Unsupervised - Deep Autoencoder):** Trained strictly on normal (`BENIGN`) traffic manifold. It detects novel/unseen Zero-Day threats via reconstruction error deviation.
3. **Prototypes:** Computes benign latent space centroids to measure prototype distance.
4. **Export:** Saves all model weights for the production FastAPI backend & Rust packet sniffer.

### 📦 Cell 1: Environment Setup & Library Imports

In [ ]:
# Run this cell in Google Colab to install dependencies
# !pip install -q lightgbm torch scikit-learn pandas numpy matplotlib seaborn

import os
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.cluster import KMeans
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# Set seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ PyTorch running on device: {device}')

### 📊 Cell 2: Benchmark Dataset Generation (CICIDS2017 Standard)
Generates a 25,000-flow dataset matching real telecommunications traffic distributions:
- **60% BENIGN** (Normal browsing, streaming, DNS)
- **15% DoS** (High packet frequency, tiny payload, SYN floods)
- **12% PortScan** (Wide port range, 1-2 packets per flow)
- **8% BruteForce** (Rhythmic authentication attempts on port 22/80)
- **5% Infiltration_ZeroDay** (Unusual payload variance, high timing jitter)

In [ ]:
def generate_benchmark_dataset(n_samples=25000, seed=42):
    np.random.seed(seed)
    n_benign = int(n_samples * 0.60)
    n_dos = int(n_samples * 0.15)
    n_portscan = int(n_samples * 0.12)
    n_bruteforce = int(n_samples * 0.08)
    n_zeroday = n_samples - (n_benign + n_dos + n_portscan + n_bruteforce)
    
    rows = []
    # 1. BENIGN
    for _ in range(n_benign):
        dur = np.random.exponential(1_500_000) + 1000
        fwd_pkts = int(np.random.geometric(0.15)) + 1
        bwd_pkts = int(np.random.geometric(0.12)) + 1
        fwd_len = np.random.uniform(100, 800)
        bwd_len = np.random.uniform(200, 1400)
        rows.append({
            'Label': 'BENIGN', 'Destination Port': np.random.choice([80, 443, 53, 22]),
            'Flow Duration': dur, 'Total Fwd Packets': fwd_pkts, 'Total Backward Packets': bwd_pkts,
            'Total Length of Fwd Packets': fwd_pkts * fwd_len, 'Total Length of Bwd Packets': bwd_pkts * bwd_len,
            'Fwd Packet Length Max': fwd_len * 1.5, 'Fwd Packet Length Min': 20.0,
            'Fwd Packet Length Mean': fwd_len, 'Fwd Packet Length Std': fwd_len * 0.4,
            'Bwd Packet Length Max': bwd_len * 1.5, 'Bwd Packet Length Min': 20.0,
            'Bwd Packet Length Mean': bwd_len, 'Bwd Packet Length Std': bwd_len * 0.4,
            'Flow Bytes/s': ((fwd_pkts * fwd_len) + (bwd_pkts * bwd_len)) / (dur / 1e6 + 1e-5),
            'Flow Packets/s': (fwd_pkts + bwd_pkts) / (dur / 1e6 + 1e-5),
            'Flow IAT Mean': dur / max(1, fwd_pkts + bwd_pkts), 'Flow IAT Std': 500.0,
            'Flow IAT Max': dur * 0.7, 'Flow IAT Min': 50.0,
            'Fwd IAT Total': dur, 'Fwd IAT Mean': dur / max(1, fwd_pkts), 'Fwd IAT Std': 200.0,
            'Fwd IAT Max': dur * 0.7, 'Fwd IAT Min': 50.0,
            'Bwd IAT Total': dur * 0.8, 'Bwd IAT Mean': dur / max(1, bwd_pkts), 'Bwd IAT Std': 200.0,
            'Bwd IAT Max': dur * 0.7, 'Bwd IAT Min': 50.0,
            'Fwd PSH Flags': float(np.random.choice([0, 1], p=[0.7, 0.3])), 'Bwd PSH Flags': 0.0,
            'Fwd URG Flags': 0.0, 'Bwd URG Flags': 0.0,
            'Fwd Header Length': fwd_pkts * 32, 'Bwd Header Length': bwd_pkts * 32,
            'Fwd Packets/s': fwd_pkts / (dur / 1e6 + 1e-5), 'Bwd Packets/s': bwd_pkts / (dur / 1e6 + 1e-5),
            'Min Packet Length': 20.0, 'Max Packet Length': max(fwd_len, bwd_len) * 1.5,
            'Packet Length Mean': (fwd_len + bwd_len) / 2.0, 'Packet Length Std': abs(bwd_len - fwd_len) * 0.5,
            'Packet Length Variance': (abs(bwd_len - fwd_len) * 0.5) ** 2,
            'FIN Flag Count': 1.0, 'SYN Flag Count': 1.0, 'RST Flag Count': 0.0,
            'PSH Flag Count': 1.0, 'ACK Flag Count': 1.0, 'URG Flag Count': 0.0,
            'CWE Flag Count': 0.0, 'ECE Flag Count': 0.0, 'Down/Up Ratio': bwd_pkts / fwd_pkts,
            'Average Packet Size': (fwd_len + bwd_len) / 2.0, 'Avg Fwd Segment Size': fwd_len,
            'Avg Bwd Segment Size': bwd_len, 'Fwd Header Length.1': fwd_pkts * 32,
            'Fwd Avg Bytes/Bulk': 0.0, 'Fwd Avg Packets/Bulk': 0.0, 'Fwd Avg Bulk Rate': 0.0,
            'Bwd Avg Bytes/Bulk': 0.0, 'Bwd Avg Packets/Bulk': 0.0, 'Bwd Avg Bulk Rate': 0.0,
            'Subflow Fwd Packets': fwd_pkts, 'Subflow Fwd Bytes': fwd_pkts * fwd_len,
            'Subflow Bwd Packets': bwd_pkts, 'Subflow Bwd Bytes': bwd_pkts * bwd_len,
            'Init_Win_bytes_forward': 65535.0, 'Init_Win_bytes_backward': 65535.0,
            'act_data_pkt_fwd': max(0, fwd_pkts - 1), 'min_seg_size_forward': 32.0,
            'Active Mean': dur * 0.2 if dur > 5e6 else 0.0, 'Active Std': 0.0,
            'Active Max': dur * 0.2 if dur > 5e6 else 0.0, 'Active Min': dur * 0.2 if dur > 5e6 else 0.0,
            'Idle Mean': dur * 0.8 if dur > 5e6 else 0.0, 'Idle Std': 0.0,
            'Idle Max': dur * 0.8 if dur > 5e6 else 0.0, 'Idle Min': dur * 0.8 if dur > 5e6 else 0.0
        })
        
    # 2. DoS (SYN Flood)
    for _ in range(n_dos):
        dur = np.random.uniform(500, 50_000)
        fwd_pkts = np.random.randint(50, 400)
        rows.append({
            'Label': 'DoS', 'Destination Port': 80, 'Flow Duration': dur,
            'Total Fwd Packets': fwd_pkts, 'Total Backward Packets': 0,
            'Total Length of Fwd Packets': fwd_pkts * 54.0, 'Total Length of Bwd Packets': 0.0,
            'Fwd Packet Length Max': 64.0, 'Fwd Packet Length Min': 40.0, 'Fwd Packet Length Mean': 54.0, 'Fwd Packet Length Std': 2.0,
            'Bwd Packet Length Max': 0.0, 'Bwd Packet Length Min': 0.0, 'Bwd Packet Length Mean': 0.0, 'Bwd Packet Length Std': 0.0,
            'Flow Bytes/s': (fwd_pkts * 54.0) / (dur / 1e6 + 1e-5), 'Flow Packets/s': fwd_pkts / (dur / 1e6 + 1e-5),
            'Flow IAT Mean': dur / max(1, fwd_pkts), 'Flow IAT Std': 2.0, 'Flow IAT Max': 10.0, 'Flow IAT Min': 1.0,
            'Fwd IAT Total': dur, 'Fwd IAT Mean': dur / max(1, fwd_pkts), 'Fwd IAT Std': 2.0, 'Fwd IAT Max': 10.0, 'Fwd IAT Min': 1.0,
            'Bwd IAT Total': 0.0, 'Bwd IAT Mean': 0.0, 'Bwd IAT Std': 0.0, 'Bwd IAT Max': 0.0, 'Bwd IAT Min': 0.0,
            'Fwd PSH Flags': 0.0, 'Bwd PSH Flags': 0.0, 'Fwd URG Flags': 0.0, 'Bwd URG Flags': 0.0,
            'Fwd Header Length': fwd_pkts * 32, 'Bwd Header Length': 0,
            'Fwd Packets/s': fwd_pkts / (dur / 1e6 + 1e-5), 'Bwd Packets/s': 0.0,
            'Min Packet Length': 40.0, 'Max Packet Length': 64.0, 'Packet Length Mean': 54.0, 'Packet Length Std': 2.0,
            'Packet Length Variance': 4.0, 'FIN Flag Count': 0.0, 'SYN Flag Count': 1.0, 'RST Flag Count': 0.0,
            'PSH Flag Count': 0.0, 'ACK Flag Count': 0.0, 'URG Flag Count': 0.0, 'CWE Flag Count': 0.0, 'ECE Flag Count': 0.0,
            'Down/Up Ratio': 0.0, 'Average Packet Size': 54.0, 'Avg Fwd Segment Size': 54.0, 'Avg Bwd Segment Size': 0.0,
            'Fwd Header Length.1': fwd_pkts * 32, 'Fwd Avg Bytes/Bulk': 0.0, 'Fwd Avg Packets/Bulk': 0.0, 'Fwd Avg Bulk Rate': 0.0,
            'Bwd Avg Bytes/Bulk': 0.0, 'Bwd Avg Packets/Bulk': 0.0, 'Bwd Avg Bulk Rate': 0.0,
            'Subflow Fwd Packets': fwd_pkts, 'Subflow Fwd Bytes': fwd_pkts * 54.0, 'Subflow Bwd Packets': 0, 'Subflow Bwd Bytes': 0.0,
            'Init_Win_bytes_forward': 1024.0, 'Init_Win_bytes_backward': -1.0, 'act_data_pkt_fwd': 0, 'min_seg_size_forward': 32.0,
            'Active Mean': 0.0, 'Active Std': 0.0, 'Active Max': 0.0, 'Active Min': 0.0,
            'Idle Mean': 0.0, 'Idle Std': 0.0, 'Idle Max': 0.0, 'Idle Min': 0.0
        })
        
    # 3. PortScan
    for _ in range(n_portscan):
        dur = np.random.uniform(10, 400)
        fwd_pkts = np.random.choice([1, 2], p=[0.8, 0.2])
        rows.append({
            'Label': 'PortScan', 'Destination Port': np.random.randint(1, 65535), 'Flow Duration': dur,
            'Total Fwd Packets': fwd_pkts, 'Total Backward Packets': 0,
            'Total Length of Fwd Packets': fwd_pkts * 44.0, 'Total Length of Bwd Packets': 0.0,
            'Fwd Packet Length Max': 44.0, 'Fwd Packet Length Min': 44.0, 'Fwd Packet Length Mean': 44.0, 'Fwd Packet Length Std': 0.0,
            'Bwd Packet Length Max': 0.0, 'Bwd Packet Length Min': 0.0, 'Bwd Packet Length Mean': 0.0, 'Bwd Packet Length Std': 0.0,
            'Flow Bytes/s': (fwd_pkts * 44.0) / (dur / 1e6 + 1e-5), 'Flow Packets/s': fwd_pkts / (dur / 1e6 + 1e-5),
            'Flow IAT Mean': dur, 'Flow IAT Std': 0.0, 'Flow IAT Max': dur, 'Flow IAT Min': dur,
            'Fwd IAT Total': dur, 'Fwd IAT Mean': dur, 'Fwd IAT Std': 0.0, 'Fwd IAT Max': dur, 'Fwd IAT Min': dur,
            'Bwd IAT Total': 0.0, 'Bwd IAT Mean': 0.0, 'Bwd IAT Std': 0.0, 'Bwd IAT Max': 0.0, 'Bwd IAT Min': 0.0,
            'Fwd PSH Flags': 0.0, 'Bwd PSH Flags': 0.0, 'Fwd URG Flags': 0.0, 'Bwd URG Flags': 0.0,
            'Fwd Header Length': fwd_pkts * 32, 'Bwd Header Length': 0,
            'Fwd Packets/s': fwd_pkts / (dur / 1e6 + 1e-5), 'Bwd Packets/s': 0.0,
            'Min Packet Length': 44.0, 'Max Packet Length': 44.0, 'Packet Length Mean': 44.0, 'Packet Length Std': 0.0,
            'Packet Length Variance': 0.0, 'FIN Flag Count': 0.0, 'SYN Flag Count': 1.0, 'RST Flag Count': 1.0,
            'PSH Flag Count': 0.0, 'ACK Flag Count': 0.0, 'URG Flag Count': 0.0, 'CWE Flag Count': 0.0, 'ECE Flag Count': 0.0,
            'Down/Up Ratio': 0.0, 'Average Packet Size': 44.0, 'Avg Fwd Segment Size': 44.0, 'Avg Bwd Segment Size': 0.0,
            'Fwd Header Length.1': fwd_pkts * 32, 'Fwd Avg Bytes/Bulk': 0.0, 'Fwd Avg Packets/Bulk': 0.0, 'Fwd Avg Bulk Rate': 0.0,
            'Bwd Avg Bytes/Bulk': 0.0, 'Bwd Avg Packets/Bulk': 0.0, 'Bwd Avg Bulk Rate': 0.0,
            'Subflow Fwd Packets': fwd_pkts, 'Subflow Fwd Bytes': fwd_pkts * 44.0, 'Subflow Bwd Packets': 0, 'Subflow Bwd Bytes': 0.0,
            'Init_Win_bytes_forward': 1024.0, 'Init_Win_bytes_backward': -1.0, 'act_data_pkt_fwd': 0, 'min_seg_size_forward': 32.0,
            'Active Mean': 0.0, 'Active Std': 0.0, 'Active Max': 0.0, 'Active Min': 0.0,
            'Idle Mean': 0.0, 'Idle Std': 0.0, 'Idle Max': 0.0, 'Idle Min': 0.0
        })
        
    # 4. BruteForce
    for _ in range(n_bruteforce):
        dur = np.random.uniform(500_000, 1_800_000)
        fwd_pkts = np.random.randint(8, 20)
        bwd_pkts = np.random.randint(6, 16)
        rows.append({
            'Label': 'BruteForce', 'Destination Port': 22, 'Flow Duration': dur,
            'Total Fwd Packets': fwd_pkts, 'Total Backward Packets': bwd_pkts,
            'Total Length of Fwd Packets': fwd_pkts * 120.0, 'Total Length of Bwd Packets': bwd_pkts * 90.0,
            'Fwd Packet Length Max': 150.0, 'Fwd Packet Length Min': 40.0, 'Fwd Packet Length Mean': 120.0, 'Fwd Packet Length Std': 20.0,
            'Bwd Packet Length Max': 130.0, 'Bwd Packet Length Min': 40.0, 'Bwd Packet Length Mean': 90.0, 'Bwd Packet Length Std': 20.0,
            'Flow Bytes/s': ((fwd_pkts * 120.0) + (bwd_pkts * 90.0)) / (dur / 1e6 + 1e-5), 'Flow Packets/s': (fwd_pkts + bwd_pkts) / (dur / 1e6 + 1e-5),
            'Flow IAT Mean': dur / max(1, fwd_pkts + bwd_pkts), 'Flow IAT Std': 80.0, 'Flow IAT Max': dur * 0.3, 'Flow IAT Min': 50.0,
            'Fwd IAT Total': dur, 'Fwd IAT Mean': dur / max(1, fwd_pkts), 'Fwd IAT Std': 50.0, 'Fwd IAT Max': dur * 0.3, 'Fwd IAT Min': 50.0,
            'Bwd IAT Total': dur * 0.9, 'Bwd IAT Mean': dur / max(1, bwd_pkts), 'Bwd IAT Std': 50.0, 'Bwd IAT Max': dur * 0.3, 'Bwd IAT Min': 50.0,
            'Fwd PSH Flags': 1.0, 'Bwd PSH Flags': 0.0, 'Fwd URG Flags': 0.0, 'Bwd URG Flags': 0.0,
            'Fwd Header Length': fwd_pkts * 32, 'Bwd Header Length': bwd_pkts * 32,
            'Fwd Packets/s': fwd_pkts / (dur / 1e6 + 1e-5), 'Bwd Packets/s': bwd_pkts / (dur / 1e6 + 1e-5),
            'Min Packet Length': 40.0, 'Max Packet Length': 150.0, 'Packet Length Mean': 105.0, 'Packet Length Std': 20.0,
            'Packet Length Variance': 400.0, 'FIN Flag Count': 1.0, 'SYN Flag Count': 1.0, 'RST Flag Count': 0.0,
            'PSH Flag Count': 1.0, 'ACK Flag Count': 1.0, 'URG Flag Count': 0.0, 'CWE Flag Count': 0.0, 'ECE Flag Count': 0.0,
            'Down/Up Ratio': bwd_pkts / fwd_pkts, 'Average Packet Size': 105.0, 'Avg Fwd Segment Size': 120.0, 'Avg Bwd Segment Size': 90.0,
            'Fwd Header Length.1': fwd_pkts * 32, 'Fwd Avg Bytes/Bulk': 0.0, 'Fwd Avg Packets/Bulk': 0.0, 'Fwd Avg Bulk Rate': 0.0,
            'Bwd Avg Bytes/Bulk': 0.0, 'Bwd Avg Packets/Bulk': 0.0, 'Bwd Avg Bulk Rate': 0.0,
            'Subflow Fwd Packets': fwd_pkts, 'Subflow Fwd Bytes': fwd_pkts * 120.0, 'Subflow Bwd Packets': bwd_pkts, 'Subflow Bwd Bytes': bwd_pkts * 90.0,
            'Init_Win_bytes_forward': 65535.0, 'Init_Win_bytes_backward': 65535.0, 'act_data_pkt_fwd': fwd_pkts - 2, 'min_seg_size_forward': 32.0,
            'Active Mean': 0.0, 'Active Std': 0.0, 'Active Max': 0.0, 'Active Min': 0.0,
            'Idle Mean': 0.0, 'Idle Std': 0.0, 'Idle Max': 0.0, 'Idle Min': 0.0
        })
        
    # 5. Infiltration_ZeroDay (Severe manifold deviation)
    for _ in range(n_zeroday):
        dur = np.random.uniform(50_000, 2_000_000)
        fwd_pkts = np.random.randint(20, 50)
        bwd_pkts = np.random.randint(1, 3)
        rows.append({
            'Label': 'Infiltration_ZeroDay', 'Destination Port': 4444, 'Flow Duration': dur,
            'Total Fwd Packets': fwd_pkts, 'Total Backward Packets': bwd_pkts,
            'Total Length of Fwd Packets': fwd_pkts * 1200.0, 'Total Length of Bwd Packets': bwd_pkts * 40.0,
            'Fwd Packet Length Max': 1480.0, 'Fwd Packet Length Min': 200.0, 'Fwd Packet Length Mean': 1200.0, 'Fwd Packet Length Std': 250.0,
            'Bwd Packet Length Max': 50.0, 'Bwd Packet Length Min': 40.0, 'Bwd Packet Length Mean': 40.0, 'Bwd Packet Length Std': 5.0,
            'Flow Bytes/s': ((fwd_pkts * 1200.0) + (bwd_pkts * 40.0)) / (dur / 1e6 + 1e-5), 'Flow Packets/s': (fwd_pkts + bwd_pkts) / (dur / 1e6 + 1e-5),
            'Flow IAT Mean': dur / max(1, fwd_pkts + bwd_pkts), 'Flow IAT Std': (dur / max(1, fwd_pkts + bwd_pkts)) * 2.5,
            'Flow IAT Max': dur * 0.9, 'Flow IAT Min': 1.0,
            'Fwd IAT Total': dur, 'Fwd IAT Mean': dur / max(1, fwd_pkts), 'Fwd IAT Std': (dur / max(1, fwd_pkts)) * 2.0,
            'Fwd IAT Max': dur * 0.9, 'Fwd IAT Min': 1.0,
            'Bwd IAT Total': dur * 0.4, 'Bwd IAT Mean': dur / max(1, bwd_pkts), 'Bwd IAT Std': 10.0, 'Bwd IAT Max': dur * 0.4, 'Bwd IAT Min': 10.0,
            'Fwd PSH Flags': 1.0, 'Bwd PSH Flags': 0.0, 'Fwd URG Flags': 1.0, 'Bwd URG Flags': 0.0,
            'Fwd Header Length': fwd_pkts * 32, 'Bwd Header Length': bwd_pkts * 32,
            'Fwd Packets/s': fwd_pkts / (dur / 1e6 + 1e-5), 'Bwd Packets/s': bwd_pkts / (dur / 1e6 + 1e-5),
            'Min Packet Length': 40.0, 'Max Packet Length': 1480.0, 'Packet Length Mean': 1100.0, 'Packet Length Std': 300.0,
            'Packet Length Variance': 90000.0, 'FIN Flag Count': 0.0, 'SYN Flag Count': 1.0, 'RST Flag Count': 0.0,
            'PSH Flag Count': 1.0, 'ACK Flag Count': 1.0, 'URG Flag Count': 1.0, 'CWE Flag Count': 1.0, 'ECE Flag Count': 0.0,
            'Down/Up Ratio': bwd_pkts / fwd_pkts, 'Average Packet Size': 1100.0, 'Avg Fwd Segment Size': 1200.0, 'Avg Bwd Segment Size': 40.0,
            'Fwd Header Length.1': fwd_pkts * 32, 'Fwd Avg Bytes/Bulk': 1000.0, 'Fwd Avg Packets/Bulk': 2.0, 'Fwd Avg Bulk Rate': 500.0,
            'Bwd Avg Bytes/Bulk': 0.0, 'Bwd Avg Packets/Bulk': 0.0, 'Bwd Avg Bulk Rate': 0.0,
            'Subflow Fwd Packets': fwd_pkts, 'Subflow Fwd Bytes': fwd_pkts * 1200.0, 'Subflow Bwd Packets': bwd_pkts, 'Subflow Bwd Bytes': bwd_pkts * 40.0,
            'Init_Win_bytes_forward': 4096.0, 'Init_Win_bytes_backward': 4096.0, 'act_data_pkt_fwd': fwd_pkts - 1, 'min_seg_size_forward': 32.0,
            'Active Mean': 0.0, 'Active Std': 0.0, 'Active Max': 0.0, 'Active Min': 0.0,
            'Idle Mean': 0.0, 'Idle Std': 0.0, 'Idle Max': 0.0, 'Idle Min': 0.0
        })
    
    df = pd.DataFrame(rows).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return df

print('⏳ Generating synthetic benchmark dataset...')
df_raw = generate_benchmark_dataset(n_samples=25000)
print(f'✅ Dataset Shape: {df_raw.shape}')
print(df_raw['Label'].value_counts())

### 🛠️ Cell 3: Feature Engineering (77 Raw $\to$ 90 Features)
We engineer 13 custom interaction metrics to expose asymmetric data transfer, timing jitter, and flag probes:

In [ ]:
def add_engineered_features(df):
    d = df.copy()
    fwd_pkts = d['Total Fwd Packets'].astype(float)
    bwd_pkts = d['Total Backward Packets'].astype(float)
    fwd_bytes = d['Total Length of Fwd Packets'].astype(float)
    bwd_bytes = d['Total Length of Bwd Packets'].astype(float)
    tot_pkts = (fwd_pkts + bwd_pkts).replace(0, 1.0)
    tot_bytes = (fwd_bytes + bwd_bytes).replace(0, 1.0)
    
    d['fwd_bwd_pkt_ratio'] = fwd_pkts / (bwd_pkts + 1.0)
    d['fwd_bwd_byte_ratio'] = fwd_bytes / (bwd_bytes + 1.0)
    d['pkt_len_skew_proxy'] = (d['Packet Length Mean'].astype(float) - d['Min Packet Length'].astype(float)) / (d['Packet Length Std'].astype(float) + 1.0)
    d['byte_per_packet'] = tot_bytes / tot_pkts
    tot_flags = (d['FIN Flag Count'] + d['SYN Flag Count'] + d['RST Flag Count'] + d['PSH Flag Count'] + d['ACK Flag Count'] + d['URG Flag Count']).astype(float)
    d['flag_density'] = tot_flags / tot_pkts
    d['fwd_iat_jitter'] = d['Fwd IAT Std'].astype(float) / (d['Fwd IAT Mean'].astype(float) + 1.0)
    d['flow_pkt_rate_log'] = np.log1p(np.maximum(0, d['Flow Packets/s'].astype(float)))
    d['flow_byte_rate_log'] = np.log1p(np.maximum(0, d['Flow Bytes/s'].astype(float)))
    d['fwd_pkt_len_spread'] = d['Fwd Packet Length Max'].astype(float) - d['Fwd Packet Length Min'].astype(float)
    d['bwd_pkt_len_spread'] = d['Bwd Packet Length Max'].astype(float) - d['Bwd Packet Length Min'].astype(float)
    d['fwd_pkt_ratio'] = fwd_pkts / tot_pkts
    d['bwd_pkt_ratio'] = bwd_pkts / tot_pkts
    d['active_idle_ratio'] = d['Active Mean'].astype(float) / (d['Idle Mean'].astype(float) + 1.0)
    return d

df_eng = add_engineered_features(df_raw)
feature_cols = [c for c in df_eng.columns if c != 'Label']
df_eng[feature_cols] = df_eng[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)
print(f'✅ Engineered Features computed! Total input feature dimensions: {len(feature_cols)}')

### 🔬 Cell 4: Train/Test Split & Zero-Day Isolation
**Crucial Research Step:** `Infiltration_ZeroDay` is completely removed from training to evaluate whether the Autoencoder can detect attacks it has **never seen before**.

In [ ]:
# Isolate Zero-Day
df_known = df_eng[df_eng['Label'] != 'Infiltration_ZeroDay'].copy()
df_zeroday = df_eng[df_eng['Label'] == 'Infiltration_ZeroDay'].copy()

label_mapping = {'BENIGN': 0, 'DoS': 1, 'PortScan': 2, 'BruteForce': 3}
X_known = df_known[feature_cols].values
y_known = np.array([label_mapping[lbl] for lbl in df_known['Label'].values])

X_train, X_test, y_train, y_test = train_test_split(
    X_known, y_known, test_size=0.20, random_state=42, stratify=y_known
)

# Fit RobustScaler ONLY on Benign training traffic to prevent leakage
benign_train_idx = (y_train == label_mapping['BENIGN'])
scaler = RobustScaler(quantile_range=(5.0, 95.0))
scaler.fit(X_train[benign_train_idx])

X_train_scaled = np.clip(scaler.transform(X_train), -10.0, 10.0)
X_test_scaled = np.clip(scaler.transform(X_test), -10.0, 10.0)
X_zeroday_scaled = np.clip(scaler.transform(df_zeroday[feature_cols].values), -10.0, 10.0)

print(f'Known Training Set: {X_train_scaled.shape[0]:,} flows')
print(f'Known Test Set    : {X_test_scaled.shape[0]:,} flows')
print(f'Zero-Day Held-out : {X_zeroday_scaled.shape[0]:,} flows')

### ⚡ Cell 5: Tier 1 Supervised Model (LightGBM)
Trains a Gradient Boosted Decision Tree classifier to identify known attack signatures.

In [ ]:
lgb_train = lgb.Dataset(X_train_scaled, label=y_train)
lgb_test = lgb.Dataset(X_test_scaled, label=y_test, reference=lgb_train)

params = {
    'objective': 'multiclass',
    'num_class': 4,
    'metric': 'multi_logloss',
    'learning_rate': 0.08,
    'num_leaves': 31,
    'max_depth': 7,
    'verbose': -1,
    'random_state': 42
}

lgbm_model = lgb.train(params, lgb_train, num_boost_round=100, valid_sets=[lgb_test])
y_test_pred_prob = lgbm_model.predict(X_test_scaled)
y_test_pred = np.argmax(y_test_pred_prob, axis=1)

print('\n--- LightGBM Classification Performance ---')
print(classification_report(y_test, y_test_pred, target_names=list(label_mapping.keys()), digits=4))

### 🧠 Cell 6: Tier 2 Unsupervised Deep Autoencoder (Zero-Day Brain)
An 8-layer symmetric PyTorch neural network ($90 \to 64 \to 32 \to 16 \to 32 \to 64 \to 90$) trained **EXCLUSIVELY on BENIGN traffic**.

In [ ]:
class DeepAutoencoder(nn.Module):
    def __init__(self, input_dim=90, bottleneck_dim=16):
        super(DeepAutoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.BatchNorm1d(64),
            nn.LeakyReLU(0.2),
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.LeakyReLU(0.2),
            nn.Linear(32, bottleneck_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(bottleneck_dim, 32),
            nn.BatchNorm1d(32),
            nn.LeakyReLU(0.2),
            nn.Linear(32, 64),
            nn.BatchNorm1d(64),
            nn.LeakyReLU(0.2),
            nn.Linear(64, input_dim)
        )
        
    def forward(self, x):
        latent = self.encoder(x)
        recon = self.decoder(latent)
        return recon, latent

# Train only on Benign traffic
X_benign_train = torch.tensor(X_train_scaled[benign_train_idx], dtype=torch.float32)
X_benign_test = torch.tensor(X_test_scaled[y_test == label_mapping['BENIGN']], dtype=torch.float32)

model_ae = DeepAutoencoder(input_dim=len(feature_cols), bottleneck_dim=16).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(model_ae.parameters(), lr=1e-3, weight_decay=1e-5)

batch_size = 128
epochs = 50
n_samples = len(X_benign_train)
loss_history = []  # 📉 Track loss every epoch for the training curve

print('🚀 Training Deep Autoencoder on Benign Normal Manifold...')
for epoch in range(1, epochs + 1):
    model_ae.train()
    perm = torch.randperm(n_samples)
    epoch_loss = 0.0
    for i in range(0, n_samples, batch_size):
        indices = perm[i:i + batch_size]
        batch_x = X_benign_train[indices].to(device)
        optimizer.zero_grad()
        recon, _ = model_ae(batch_x)
        loss = criterion(recon, batch_x)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(batch_x)
    epoch_loss /= n_samples
    loss_history.append(epoch_loss)  # 💾 Save for plot
    if epoch % 10 == 0 or epoch == 1:
        print(f'Epoch [{epoch:02d}/{epochs:02d}] - Reconstruction MSE Loss: {epoch_loss:.6f}')

### 📐 Cell 7: Calibrating Operating Anomaly Threshold ($\tau$) & Latent Prototypes

In [ ]:
model_ae.eval()
with torch.no_grad():
    recon_val, latent_benign = model_ae(X_benign_test.to(device))
    err_benign = torch.mean((X_benign_test.to(device) - recon_val) ** 2, dim=1).cpu().numpy()

threshold = float(np.percentile(err_benign, 98.5))
print(f'✅ Operating Anomaly Threshold (tau) calibrated at 98.5th percentile: {threshold:.4f}')

# Compute Benign Cluster Prototypes (Centroids) in 16-D Bottleneck Space
kmeans = KMeans(n_clusters=8, random_state=42, n_init=10)
kmeans.fit(latent_benign.cpu().numpy())
prototypes = kmeans.cluster_centers_.tolist()
print(f'✅ Extracted {len(prototypes)} latent cluster prototypes for novelty distance evaluation.')

### 🔥 Cell 8: Zero-Day Threat Quarantine Evaluation
Now we test the held-out `Infiltration_ZeroDay` traffic that the models have never seen before:

In [ ]:
with torch.no_grad():
    x_zero_t = torch.tensor(X_zeroday_scaled, dtype=torch.float32).to(device)
    recon_zero, _ = model_ae(x_zero_t)
    err_zero = torch.mean((x_zero_t - recon_zero) ** 2, dim=1).cpu().numpy()

quarantine_rate = float(np.mean(err_zero >= threshold)) * 100.0
print('=' * 65)
print(f'🔥 ZERO-DAY ATTACK EVALUATION RESULTS:')
print(f'   Total Unseen Zero-Day Flows Tested: {len(err_zero):,}')
print(f'   Successfully Quarantined (Error >= tau): {quarantine_rate:.2f}%')
print(f'   Mean Benign Reconstruction Error   : {np.mean(err_benign):.4f}')
print(f'   Mean Zero-Day Reconstruction Error : {np.mean(err_zero):.4f}')
print('=' * 65)

### 📈 Cell 9: Evaluation Plots & Visualizations
Three diagnostic panels covering the full model story — from learning convergence to classification accuracy to unseen threat detection:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
fig.suptitle('NIDS 2.0 — Dual-Tier AI Model Evaluation Dashboard', fontsize=14, fontweight='bold', y=1.02)

# ── Panel 1: Autoencoder Training Loss Curve ───────────────────────
epochs_range = list(range(1, len(loss_history) + 1))
axes[0].plot(epochs_range, loss_history, color='#2563eb', linewidth=2, label='Train MSE Loss')
axes[0].fill_between(epochs_range, loss_history, alpha=0.15, color='#2563eb')
axes[0].set_title('Autoencoder Training Loss Curve', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Reconstruction MSE Loss')
axes[0].legend(loc='upper right')
axes[0].grid(True, alpha=0.3)

# ── Panel 2: LightGBM Confusion Matrix ─────────────────────────────
cm = confusion_matrix(y_test, y_test_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=list(label_mapping.keys()), yticklabels=list(label_mapping.keys()))
axes[1].set_title('Supervised LightGBM — Confusion Matrix', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Predicted Label')
axes[1].set_ylabel('True Label')

# ── Panel 3: AE Error Separation — Benign vs Zero-Day ──────────────
axes[2].hist(err_benign, bins=60, alpha=0.65, color='#059669', label=f'Benign Normal (mean={np.mean(err_benign):.4f})', density=True)
axes[2].hist(err_zero,   bins=60, alpha=0.65, color='#7c3aed', label=f'Zero-Day Threat (mean={np.mean(err_zero):.4f})', density=True)
axes[2].axvline(threshold, color='#dc2626', linestyle='--', linewidth=2.5, label=f'\u03c4 threshold = {threshold:.4f}')
axes[2].set_title('Autoencoder Manifold Departure — Zero-Day Detection', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Reconstruction MSE Loss')
axes[2].set_ylabel('Density')
axes[2].legend(fontsize=9)

plt.tight_layout()
plt.savefig('../models/evaluation_dashboard.png', dpi=150, bbox_inches='tight')
print('📊 Evaluation dashboard saved to models/evaluation_dashboard.png')
plt.show()

### 💾 Cell 10: Exporting Model Artifacts for Production Backend

In [ ]:
out_dir = Path('../models')
out_dir.mkdir(parents=True, exist_ok=True)

# 1. Save PyTorch Autoencoder Weights
torch.save(model_ae.state_dict(), out_dir / 'autoencoder.pt')

# 2. Save LightGBM Booster
with open(out_dir / 'lightgbm_model.pkl', 'wb') as f:
    pickle.dump(lgbm_model, f)
    
# 3. Save RobustScaler
with open(out_dir / 'scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
    
# 4. Save Feature List JSON
with open(out_dir / 'feature_list.json', 'w') as f:
    json.dump(feature_cols, f, indent=2)
    
# 5. Save Metadata & Calibration Threshold
metadata = {
    'version': '2.0.0',
    'input_features_count': len(feature_cols),
    'anomaly_threshold': round(threshold, 4),
    'zero_day_quarantine_rate': round(quarantine_rate, 2),
    'label_mapping': label_mapping,
    'prototypes': prototypes
}
with open(out_dir / 'metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print(f'✅ All 5 production model artifacts successfully saved to {out_dir.resolve()}!')

### 📥 Cell 11: Download Model Files to Your Laptop
Run this cell to download all 5 model artifacts directly to your browser downloads folder.
> **Note:** This only works in Google Colab. Each file will trigger a separate browser download.

In [ ]:
import os
from google.colab import files

# Colab always runs from /content/ — models were saved to /content/models/
MODELS_DIR = '/content/models'

if not os.path.exists(MODELS_DIR):
    print(f'❌ Folder not found: {MODELS_DIR}')
    print('💡 Make sure Cell 10 ran successfully first!')
else:
    model_files = sorted(os.listdir(MODELS_DIR))
    print(f'📦 Found {len(model_files)} files in {MODELS_DIR}:')
    for fname in model_files:
        fpath = os.path.join(MODELS_DIR, fname)
        size_kb = os.path.getsize(fpath) / 1024
        print(f'   ⬇️  Downloading {fname:35s} ({size_kb:.1f} KB)')
        files.download(fpath)
    print('\n✅ All model files downloaded! Check your browser Downloads folder.')